# Лабораторная работа №3
## Плоская кинематическая пара, циклоидальные кривые и комплексные экспоненты

В этой работе используются плоские вращения на плоскости и их запись через матрицы и комплексные числа.
Ноутбук разделен по заданиям из `03.pdf`: сначала рассматривается плоская кинематическая пара, затем на ее
основе строятся эпитрохоиды и гипотрохоиды, после этого добавляется интерактивная визуализация движения,
а в конце изучаются кривые, задаваемые суммами комплексных экспонент.


In [ ]:
import math
import numpy as np
import matplotlib.patches as mpatches
from ipywidgets import interact
from matplotlib.figure import Figure


DPI = 200
FIGSIZE = (6.0, 6.0)


def prepare_axes(ax, xlim, ylim, title=None):
    ax.set_xlim(xlim[0], xlim[1])
    ax.set_ylim(ylim[0], ylim[1])
    ax.set_aspect("equal")
    ax.grid(visible=True, color="0.85", linewidth=0.8, linestyle="--")
    ax.set_xlabel("x")
    ax.set_ylabel("y")
    if title is not None:
        ax.set_title(title)


def make_figure(xlim, ylim, figsize=FIGSIZE, dpi=DPI, title=None):
    fig = Figure(figsize=figsize, dpi=dpi, layout="constrained")
    ax = fig.add_subplot(1, 1, 1)
    prepare_axes(ax, xlim=xlim, ylim=ylim, title=title)
    return fig, ax


def padded_limits(points, pad_ratio=0.10, min_pad=0.50):
    points = np.asarray(points, dtype=np.float64)
    xmin = float(np.min(points[:, 0]))
    xmax = float(np.max(points[:, 0]))
    ymin = float(np.min(points[:, 1]))
    ymax = float(np.max(points[:, 1]))
    span = max(xmax - xmin, ymax - ymin)
    pad = max(min_pad, pad_ratio * span)
    return (xmin - pad, xmax + pad), (ymin - pad, ymax + pad)


def draw_point(ax, point, label=None, color="black", dx=0.08, dy=0.08, size=28):
    ax.scatter([point[0]], [point[1]], color=color, s=size, zorder=5)
    if label is not None:
        ax.text(point[0] + dx, point[1] + dy, label, color=color)


def draw_arrow(ax, start, end, color="tab:blue", label=None, linestyle="-", linewidth=1.8):
    start = np.asarray(start, dtype=np.float64)
    end = np.asarray(end, dtype=np.float64)
    ax.annotate(
        "",
        xy=end,
        xytext=start,
        arrowprops={
            "arrowstyle": "->",
            "color": color,
            "linewidth": linewidth,
            "linestyle": linestyle,
            "shrinkA": 0.0,
            "shrinkB": 0.0,
        },
    )
    if label is not None:
        midpoint = 0.5 * (start + end)
        ax.text(midpoint[0] + 0.08, midpoint[1] + 0.08, label, color=color)


def complex_to_points(z_values):
    z_values = np.asarray(z_values, dtype=np.complex128)
    return np.column_stack((z_values.real, z_values.imag))


## Задание 1. Плоская кинематическая пара

Плоская кинематическая пара, изображенная в презентации, является упрощенной моделью плоского двухзвенного
манипулятора. Слово *кинематическая* означает, что нас интересует именно геометрия движения: длины звеньев,
углы поворота и траектория конечной точки, а не силы, массы и уравнения динамики.

Геометрический смысл параметров следующий:

- $a_1$ и $a_2$ — длины первого и второго звена;
- $\theta_1$ — угол поворота первого звена относительно оси $Ox$;
- $\theta_2$ — относительный угол поворота второго звена относительно первого звена;
- искомой обычно является конечная точка второго звена.

В прямой кинематической задаче величины $a_1$, $a_2$, $\theta_1$, $\theta_2$ задаются, а координаты точки
конца манипулятора вычисляются. Такая модель может служить упрощением для роботизированной руки, локтевого
механизма, стрелы крана, бедра и голени в биомеханике, а также для различных шарнирных механизмов.

При основании в начале координат конечная точка имеет координаты
$$
\mathbf{p}(\theta_1, \theta_2)=
\begin{pmatrix}
a_1 \cos \theta_1 + a_2 \cos(\theta_1 + \theta_2) \\
a_1 \sin \theta_1 + a_2 \sin(\theta_1 + \theta_2)
\end{pmatrix}.
$$


In [ ]:
def kinematic_pair_endpoint(theta1, theta2, a1, a2):
    theta1 = np.asarray(theta1, dtype=np.float64)
    theta2 = np.asarray(theta2, dtype=np.float64)
    x = a1 * np.cos(theta1) + a2 * np.cos(theta1 + theta2)
    y = a1 * np.sin(theta1) + a2 * np.sin(theta1 + theta2)
    if x.ndim == 0:
        return np.array([float(x), float(y)], dtype=np.float64)
    return np.column_stack((x, y))


def kinematic_pair_points(theta1, theta2, a1, a2):
    base = np.array([0.0, 0.0], dtype=np.float64)
    joint = np.array(
        [a1 * np.cos(theta1), a1 * np.sin(theta1)],
        dtype=np.float64,
    )
    end = joint + np.array(
        [a2 * np.cos(theta1 + theta2), a2 * np.sin(theta1 + theta2)],
        dtype=np.float64,
    )
    return base, joint, end


def plot_kinematic_pair(theta1_deg=35.0, theta2_deg=55.0, a1=3.0, a2=2.0):
    theta1 = np.deg2rad(theta1_deg)
    theta2 = np.deg2rad(theta2_deg)
    base, joint, end = kinematic_pair_points(theta1, theta2, a1, a2)

    radius = a1 + a2 + 1.0
    fig, ax = make_figure(
        xlim=(-radius, radius),
        ylim=(-radius, radius),
        title=f"Плоская кинематическая пара: theta1 = {theta1_deg:.0f}°, theta2 = {theta2_deg:.0f}°",
    )

    ax.plot([0.0, a1, a1 + a2], [0.0, 0.0, 0.0], color="0.82", linestyle=":", linewidth=1.5)
    ax.plot([base[0], joint[0]], [base[1], joint[1]], color="tab:blue", linewidth=2.2)
    ax.plot([joint[0], end[0]], [joint[1], end[1]], color="tab:orange", linewidth=2.2)

    draw_point(ax, base, label="O", color="black")
    draw_point(ax, joint, label="P1", color="tab:blue")
    draw_point(ax, end, label="P2", color="crimson")

    midpoint_1 = 0.5 * (base + joint)
    midpoint_2 = 0.5 * (joint + end)
    ax.text(midpoint_1[0] + 0.08, midpoint_1[1] + 0.08, f"a1 = {a1:.1f}", color="tab:blue")
    ax.text(midpoint_2[0] + 0.08, midpoint_2[1] + 0.08, f"a2 = {a2:.1f}", color="tab:orange")
    ax.text(end[0] + 0.12, end[1] - 0.30, f"({end[0]:.2f}, {end[1]:.2f})", color="crimson")

    return fig


In [ ]:
plot_kinematic_pair(theta1_deg=35.0, theta2_deg=55.0, a1=3.0, a2=2.0)


In [ ]:
interactive_pair = interact(
    lambda theta1_deg, theta2_deg: plot_kinematic_pair(theta1_deg=theta1_deg, theta2_deg=theta2_deg, a1=3.0, a2=2.0),
    theta1_deg=(-180, 180, 1),
    theta2_deg=(-180, 180, 1),
)
pair_output = interactive_pair.widget.children[-1]
pair_output.layout.height = "720px"
interactive_pair


## Задание 2. Циклоидальные кривые

Напомним определения:

- **циклоида** — траектория точки на окружности, катящейся без скольжения по прямой;
- **эпитрохоида** — траектория точки, жестко связанной с окружностью радиуса $r$, катящейся снаружи окружности радиуса $R$;
- **гипотрохоида** — траектория точки, жестко связанной с окружностью радиуса $r$, катящейся внутри окружности радиуса $R$;
- **эпициклоида** — частный случай эпитрохоиды при $d = r$;
- **гипоциклоида** — частный случай гипотрохоиды при $d = r$.

Связь параметров циклоидальных кривых с параметрами кинематической пары берется из презентации:

- для эпитрохоиды: $a_1 = R + r$, $a_2 = d$, $\theta_1 = t$, $\theta_2 = \dfrac{R}{r} t + \pi$;
- для гипотрохоиды: $a_1 = R - r$, $a_2 = d$, $\theta_1 = t$, $\theta_2 = - \dfrac{R}{r} t$.

Тем самым отдельные формулы не нужны: достаточно использовать уже написанную функцию для кинематической пары.


In [ ]:
def closed_period(R, r):
    R_int = int(round(R))
    r_int = int(round(r))
    if not np.isclose(R, R_int) or not np.isclose(r, r_int):
        raise ValueError("Для вычисления периода здесь используются целочисленные R и r.")
    return 2.0 * np.pi * math.lcm(abs(R_int), abs(r_int)) / abs(R_int)


def epitrochoid_points(t_values, R, r, d):
    theta1 = np.asarray(t_values, dtype=np.float64)
    theta2 = (R / r) * theta1 + np.pi
    return kinematic_pair_endpoint(theta1, theta2, a1=R + r, a2=d)


def hypotrochoid_points(t_values, R, r, d):
    theta1 = np.asarray(t_values, dtype=np.float64)
    theta2 = -(R / r) * theta1
    return kinematic_pair_endpoint(theta1, theta2, a1=R - r, a2=d)


def epicycloid_points(t_values, R, r):
    return epitrochoid_points(t_values, R=R, r=r, d=r)


def hypocycloid_points(t_values, R, r):
    return hypotrochoid_points(t_values, R=R, r=r, d=r)


def plot_trochoid_examples():
    examples = [
        ("Гипоциклоида (дельтоида)", "hypo", 3, 1, 1),
        ("Гипоциклоида (астроида)", "hypo", 4, 1, 1),
        ("Гипотрохоида", "hypo", 5, 3, 5),
        ("Эпициклоида (кардиоида)", "epi", 1, 1, 1),
        ("Эпициклоида (нефроида)", "epi", 2, 1, 1),
        ("Эпитрохоида", "epi", 2, 1, 3),
    ]

    fig = Figure(figsize=(12.0, 8.0), dpi=DPI, layout="constrained")
    axes = np.asarray(fig.subplots(2, 3))

    for ax, (name, kind, R, r, d) in zip(axes.flat, examples):
        t_values = np.linspace(0.0, closed_period(R, r), 1600)
        if kind == "hypo":
            curve = hypotrochoid_points(t_values, R=R, r=r, d=d)
        else:
            curve = epitrochoid_points(t_values, R=R, r=r, d=d)

        xlim, ylim = padded_limits(curve, pad_ratio=0.12, min_pad=0.6)
        prepare_axes(ax, xlim=xlim, ylim=ylim, title=f"{name}\nR={R}, r={r}, d={d}")
        ax.plot(curve[:, 0], curve[:, 1], color="tab:blue", linewidth=2.0)
        ax.scatter([curve[0, 0]], [curve[0, 1]], color="crimson", s=24, zorder=5)

    return fig


plot_trochoid_examples()


В частности, если взять `d = r`, то автоматически получаются эпициклоида и гипоциклоида. Это соответствует
требованию задания: частные случаи возникают из тех же самых функций без написания новых формул.


In [ ]:
TASK3_R = 5.0
TASK3_r = 3.0
TASK3_d = 5.0
TASK3_PERIOD = closed_period(TASK3_R, TASK3_r)
TASK3_BACKGROUND = hypotrochoid_points(
    np.linspace(0.0, TASK3_PERIOD, 1800),
    R=TASK3_R,
    r=TASK3_r,
    d=TASK3_d,
)


def plot_hypotrochoid_motion(t, R=TASK3_R, r=TASK3_r, d=TASK3_d):
    theta1 = t
    theta2 = -(R / r) * t
    _, center, point = kinematic_pair_points(theta1, theta2, a1=R - r, a2=d)

    xlim, ylim = padded_limits(TASK3_BACKGROUND, pad_ratio=0.20, min_pad=1.2)
    fig, ax = make_figure(
        xlim=xlim,
        ylim=ylim,
        figsize=(7.0, 7.0),
        title="Задание 3. Движение гипотрохоиды при изменении параметра t",
    )

    fixed_circle = mpatches.Circle((0.0, 0.0), radius=R, fill=False, linewidth=1.6, color="0.35")
    rolling_circle = mpatches.Circle(tuple(center), radius=r, fill=False, linewidth=1.8, color="tab:orange")
    ax.add_patch(fixed_circle)
    ax.add_patch(rolling_circle)

    ax.plot(TASK3_BACKGROUND[:, 0], TASK3_BACKGROUND[:, 1], color="0.88", linewidth=1.0)

    trace_t = np.linspace(0.0, max(t, 1.0e-6), 700)
    trace = hypotrochoid_points(trace_t, R=R, r=r, d=d)
    ax.plot(trace[:, 0], trace[:, 1], color="tab:blue", linewidth=2.2)

    ax.plot([0.0, center[0]], [0.0, center[1]], color="0.55", linestyle="--", linewidth=1.4)
    ax.plot([center[0], point[0]], [center[1], point[1]], color="tab:orange", linewidth=2.0)

    draw_point(ax, np.array([0.0, 0.0]), label="O", color="black")
    draw_point(ax, center, label="C", color="tab:orange")
    draw_point(ax, point, label="P(t)", color="crimson", dx=0.10, dy=0.10)

    ax.text(
        xlim[0] + 0.25,
        ylim[1] - 0.55,
        f"R = {R:.1f}, r = {r:.1f}, d = {d:.1f}, t = {t:.2f}",
        bbox={"facecolor": "white", "edgecolor": "0.7", "alpha": 0.9},
    )

    return fig


interactive_trochoid = interact(
    lambda t: plot_hypotrochoid_motion(t, R=TASK3_R, r=TASK3_r, d=TASK3_d),
    t=(0.0, TASK3_PERIOD, 0.02),
)
trochoid_output = interactive_trochoid.widget.children[-1]
trochoid_output.layout.height = "760px"
interactive_trochoid


## Задание 4. Кривые, задаваемые комплексными экспонентами

Комплексному числу $p(t) = x(t) + i y(t)$ сопоставляется радиус-вектор
$$
\mathbf{p}(t) =
\begin{pmatrix}
x(t) \\
y(t)
\end{pmatrix}.
$$

Для функции $p(t) = e^{it}$ получаем единичную окружность, потому что по формуле Эйлера
$$
e^{it} = \cos t + i \sin t.
$$

Для функции
$$
p(t) = e^{- \frac{\pi}{2} i t} + 3 e^{\frac{\pi}{2} i t}
$$
после группировки действительной и мнимой частей имеем
$$
p(t) = 4 \cos \frac{\pi t}{2} + 2 i \sin \frac{\pi t}{2},
$$
то есть получается эллипс с полуосями $4$ и $2$.

В качестве собственной кривой ниже выбрана сумма трех комплексных экспонент
$$
p(t) = 2.4 e^{i t} + 1.1 e^{-2 i t} + 0.8 e^{5 i t}.
$$


In [ ]:
def complex_curve(t_values, amplitudes, frequencies):
    t_values = np.asarray(t_values, dtype=np.float64)
    z = np.zeros_like(t_values, dtype=np.complex128)
    for amplitude, frequency in zip(amplitudes, frequencies):
        z = z + amplitude * np.exp(1j * frequency * t_values)
    return z


def complex_terms(t, amplitudes, frequencies):
    values = []
    for amplitude, frequency in zip(amplitudes, frequencies):
        values.append(amplitude * np.exp(1j * frequency * t))
    return np.asarray(values, dtype=np.complex128)


def plot_complex_curve_on_axes(ax, z_values, title, color="tab:blue"):
    points = complex_to_points(z_values)
    xlim, ylim = padded_limits(points, pad_ratio=0.15, min_pad=0.7)
    prepare_axes(ax, xlim=xlim, ylim=ylim, title=title)
    ax.plot(points[:, 0], points[:, 1], color=color, linewidth=2.0)
    ax.scatter([points[0, 0]], [points[0, 1]], color="crimson", s=24, zorder=5)


In [ ]:
CUSTOM_AMPLITUDES = np.array([2.4, 1.1, 0.8], dtype=np.float64)
CUSTOM_FREQUENCIES = np.array([1.0, -2.0, 5.0], dtype=np.float64)

t_circle = np.linspace(0.0, 2.0 * np.pi, 800)
z_circle = np.exp(1j * t_circle)

t_ellipse = np.linspace(0.0, 4.0, 800)
z_ellipse = np.exp(-0.5j * np.pi * t_ellipse) + 3.0 * np.exp(0.5j * np.pi * t_ellipse)

t_custom = np.linspace(0.0, 2.0 * np.pi, 1600)
z_custom = complex_curve(t_custom, CUSTOM_AMPLITUDES, CUSTOM_FREQUENCIES)

fig = Figure(figsize=(14.0, 4.6), dpi=DPI, layout="constrained")
ax_circle, ax_ellipse, ax_custom = fig.subplots(1, 3)

plot_complex_curve_on_axes(ax_circle, z_circle, title="p(t) = exp(i t)")
plot_complex_curve_on_axes(ax_ellipse, z_ellipse, title="exp(-pi i t / 2) + 3 exp(pi i t / 2)")
plot_complex_curve_on_axes(ax_custom, z_custom, title="2.4 exp(i t) + 1.1 exp(-2 i t) + 0.8 exp(5 i t)")

fig


In [ ]:
CUSTOM_CURVE = complex_curve(
    np.linspace(0.0, 2.0 * np.pi, 1800),
    CUSTOM_AMPLITUDES,
    CUSTOM_FREQUENCIES,
)


def plot_complex_vector_sum(t):
    terms = complex_terms(t, CUSTOM_AMPLITUDES, CUSTOM_FREQUENCIES)
    partial_sums = np.cumsum(terms)
    total = partial_sums[-1]

    fig = Figure(figsize=(13.0, 6.0), dpi=DPI, layout="constrained")
    ax_curve, ax_vectors = fig.subplots(1, 2)

    curve_points = complex_to_points(CUSTOM_CURVE)
    xlim_curve, ylim_curve = padded_limits(curve_points, pad_ratio=0.16, min_pad=0.8)
    prepare_axes(ax_curve, xlim=xlim_curve, ylim=ylim_curve, title="Кривая и текущее положение точки")
    ax_curve.plot(CUSTOM_CURVE.real, CUSTOM_CURVE.imag, color="0.86", linewidth=1.0)

    trace_t = np.linspace(0.0, max(t, 1.0e-6), 700)
    trace = complex_curve(trace_t, CUSTOM_AMPLITUDES, CUSTOM_FREQUENCIES)
    ax_curve.plot(trace.real, trace.imag, color="tab:blue", linewidth=2.0)
    ax_curve.scatter([total.real], [total.imag], color="crimson", s=30, zorder=5)
    ax_curve.text(total.real + 0.10, total.imag + 0.10, "p(t)", color="crimson")

    radius = float(np.sum(CUSTOM_AMPLITUDES) + 0.8)
    prepare_axes(
        ax_vectors,
        xlim=(-radius, radius),
        ylim=(-radius, radius),
        title="Сумма вращающихся векторов p1(t), p2(t), p3(t)",
    )

    start_0 = np.array([0.0, 0.0], dtype=np.float64)
    end_1 = np.array([partial_sums[0].real, partial_sums[0].imag], dtype=np.float64)
    end_2 = np.array([partial_sums[1].real, partial_sums[1].imag], dtype=np.float64)
    end_3 = np.array([partial_sums[2].real, partial_sums[2].imag], dtype=np.float64)

    draw_arrow(ax_vectors, start_0, end_1, color="tab:blue", label="p1(t)")
    draw_arrow(ax_vectors, end_1, end_2, color="tab:orange", label="p2(t)")
    draw_arrow(ax_vectors, end_2, end_3, color="tab:green", label="p3(t)")
    draw_arrow(ax_vectors, start_0, end_3, color="black", label="p(t)", linestyle="--", linewidth=1.4)

    draw_point(ax_vectors, start_0, label="O", color="black")
    draw_point(ax_vectors, end_1, color="tab:blue")
    draw_point(ax_vectors, end_2, color="tab:orange")
    draw_point(ax_vectors, end_3, color="crimson", label="P")

    ax_vectors.text(
        -radius + 0.20,
        radius - 0.45,
        f"t = {t:.2f}",
        bbox={"facecolor": "white", "edgecolor": "0.7", "alpha": 0.9},
    )

    return fig


interactive_complex = interact(
    plot_complex_vector_sum,
    t=(0.0, 2.0 * np.pi, 0.02),
)
complex_output = interactive_complex.widget.children[-1]
complex_output.layout.height = "760px"
interactive_complex
